# Compare CNN-LSTM, DonderFormer, and a matched-count random control

Run notebook 06 (or its saved-chart-only recipe) for each available model first.
Set `MODEL_RUNS` in the shared `evaluation_config.py` to absolute saved-run folders.
All settings there are **PLACEHOLDER - team to confirm**. The DonderFormer entry may
remain None until its generated frame charts exist; LSTM/random plots still work,
and significance results explicitly say `awaiting_donderformer`.

Charts are paired by chart_id, restricted to the intersection shared by all available
models and random. Exclusions and source provenance are recorded. Pooled human references
must come from the same complete test split, irrespective of missing model outputs.
The summary tables and figures include overall results and each difficulty group.


## Oni-only difficulty groups and sparse references

This dataset contains **Oni charts only**. Here, "difficulty" means **star level within Oni**.
The manifest's blank LEVEL values are recovered from the corresponding raw TJA course.
`MIN_CHARTS_PER_BAND_GROUP=30` is the provisional minimum number of training charts per group.
We merge the lowest sparse star group with its nearest adjacent group, repeat as needed,
and break equal-distance ties toward the lower-star neighbor. Only training counts choose
merges; test counts are reported, not used to choose them.

The merged mapping is shared by training bands/type distributions, pooled HI references,
and every difficulty summary. Training/test counts print below and are saved in each
`analysis/difficulty_groups.json` and `.csv`, together with original counts and merge history.
If even all available training charts cannot meet the threshold, the single remaining group
is explicitly marked below minimum. Counts exclude charts in the matching extractor failure list.


In [ ]:
from pathlib import Path
from donderformer.paths import outputs_dir
from donderformer.difficulty_groups import print_group_report
from donderformer.evaluation_config import evaluation_config
from donderformer.evaluation_comparison import load_model_results, compare_models

CONFIG = evaluation_config()
model_runs = {name: Path(folder).expanduser() if folder is not None else None
              for name, folder in CONFIG["MODEL_RUNS"].items()}
models, source_metadata = load_model_results(model_runs, CONFIG)
print_group_report(source_metadata["provenance"]["difficulty_groups"])
print("Available models:", list(models))
print("Unavailable models:", source_metadata["skipped_models"])
OUTPUT_DIR = outputs_dir() / "comparisons" / (CONFIG["PRIMARY_MODE"] + "_" + CONFIG["PRIMARY_VERSION"])


## HI precision is the primary pattern metric

A pooled human reference can contain many more unique patterns than any single chart.
Coverage (`generated ∩ pool` divided by `pool`) can therefore be tiny even for a useful chart
and can rise simply because the chart is longer. HI **precision** instead asks: what share
of this chart's unique, non-empty generated patterns appears in the human pool?
It is a fraction from 0 to 1; all-zero windows are excluded from numerator and denominator,
and a chart with no non-empty windows gets null. `HI_PRIMARY="precision"` selects it for
headline tables, plots and retention. Set it to `"coverage"` to use coverage instead.

Existing `..._hi` coverage keys remain in per-chart JSON, unchanged in percent units.
New `..._hi_coverage` labels the same percentage explicitly; `..._hi_precision` is the new
fraction. `..._hi_primary` is the configured choice expressed as a fraction.
The legacy eight-frame paper `HI_PSpace` remains unchanged.


## Overall P-Space uses the matching human chart as its scale

The possible type-pattern space grows as `7**window`, so normalized percentages approach
zero at long windows. The headline Overall value is now
`unique generated patterns / unique matching-human patterns`: self-comparison is 1,
and fewer unique patterns gives a value below 1. Both variants report this ratio at every
window as `..._overall_ratio` (the existing `..._ratio` key is retained).

Summaries show normalized percentages only for the legacy eight-frame binary case.
Old normalized `..._overall` and `..._human_overall` keys remain in per-chart JSON for
compatibility, and are omitted from the new long-window summary/comparison tables.
Overall ratios retain empty windows; the empty-window exclusion applies to HI precision.


## Random density is excluded from comparisons

Random density is excluded because the control fixes the human note count; its active span
may still differ. Its density and in-band flag are omitted from random summary tables and
all notebook 07 comparison tests/tables; Don/Ka remains. Existing per-chart and seed JSON
density keys stay available for compatibility. `summary_notes.json` records the exclusion.


## One pre-declared primary test; Holm correction for secondary tests

Choose `HI_PRIMARY`, `PRIMARY_TEST_VARIANT`, mode and version **before looking at results**.
For each chart/model, retention is HI at the **longest** configured window divided by HI at
the **shortest** window. The value is null when the shortest HI is zero/null, or the longest
is null. For random, divide the seed-averaged HI values; do not treat seeds as extra charts.

The single primary test is an overall, chart-paired, two-sided Wilcoxon signed-rank test of
retention for `PRIMARY_TEST_VARIANT="binary"`, LSTM versus DonderFormer. Its cohort is the two
models' shared valid charts, independent of missing random outputs. Higher retention is
consistent with slower HI decline; the test itself is two-sided. Primary p-values are unadjusted.

All other tests, including every per-difficulty retention test and the other variant, are
secondary. Their finite p-values form **one Holm-Bonferroni family across metrics and groups**.
This limits false positives from running many tests. Null/unavailable tests are not in the
family. The result table labels primary/secondary and reports raw and adjusted p-values;
`p_value` remains the raw compatibility key, and secondary significance uses `p_value_adjusted`.
`primary_test.json` saves the single primary result, and `retention.json` saves chart-level values.

Signed-rank tests drop zero differences and average tied ranks; chart pairs can share audio,
so independence remains a limitation. [Holm correction convention](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html).


Pattern window units: PATTERN_WINDOWS_FRAMES is measured in 23 ms frames
(8 frames ≈ 0.18 s, 64 frames ≈ 1.5 s). The proposal does not fix the units, so frames
are used for now. Measure-based windows (as in Hanzen et al., 2025) would keep windows
comparable across tempos but require beat/measure information; switch later if the
team decides to.

Held-note counting: onset and joint precision/recall now count note starts only, so a
drumroll or denden counts once rather than once per frame. Earlier versions counted
held notes per frame; onset scores from those runs are not directly comparable.
Re-run evaluation on saved charts before comparing. Greedy matching and tolerance
are unchanged. The historical OCHuman-style recall pass and eight-frame paper
DCHuman/OCHuman/Over_PSpace/HI_PSpace definitions remain unchanged.

## Main hypothesis: HI precision retention versus window length

The main figure uses the configured primary HI metric for both variants, with one line
per available model and the seed-averaged random control. Overall P-Space comparisons
use matching-human ratios; the table marks headline metrics explicitly. Tables and
figures also cover the merged star groups. No DonderFormer outputs means primary and
secondary tests remain `awaiting_donderformer` with null p-values.


In [ ]:
table, paired_tests = compare_models(models, OUTPUT_DIR, CONFIG, source_metadata)
print("Saved metric table:", OUTPUT_DIR / "metrics.csv")
print("Saved main figure:", OUTPUT_DIR / "plots" / "hi_pspace.png")
for row in table:
    if row["scope"] == "overall":
        print(row)

print("Primary test:", next(t for t in paired_tests if t["test_role"] == "primary"))
